# Middleware for LangChain Agents

This notebook explains what middleware is, why it helps control agents more tightly, and provides code examples for common middleware patterns.

## What is middleware?

Middleware is a layer of reusable code that sits between the user request and the agent or model execution. It can inspect, modify, or block requests and responses, giving you finer control over how the agent behaves.

In AI engineering, middleware is useful for logging, validation, prompt transformation, fallback behavior, and safety checks without changing the core agent logic.

## Why use middleware with agents?

Tight control of an agent means you can add cross-cutting behavior before and after each agent call. Instead of modifying the agent internals, middleware wraps agent execution and adds behavior in a composable way.

Middleware is ideal for these agent control concerns:
1. Tracking agent behavior (logging, analytics, debugging)
2. Transforming prompts, tool selection, output formatting
3. Adding retries, fallback, early termination logic
4. Applying rate limits, guardrails, and PII detection

## Middleware pattern

A simple middleware pattern wraps an agent call with pre- and post-processing logic. The agent executor is treated like a function, and middleware can be chained.

In [ ]:
from typing import Any, Callable, Dict, List

AgentFunction = Callable[[Dict[str, Any]], Dict[str, Any]]

class AgentMiddleware:
    def __init__(self, handler: AgentFunction):
        self.handler = handler

    def __call__(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        return self.handler(inputs)

def compose_middlewares(middlewares: List[AgentMiddleware], base_handler: AgentFunction) -> AgentFunction:
    handler = base_handler
    for middleware in reversed(middlewares):
        previous = handler
        handler = lambda inputs, mw=middleware, prev=previous: mw(prev)(inputs)
    return handler

## 1. Tracking agent behavior

Use middleware to log requests and responses, capture analytics, or trace debugging details. This is the first line of visibility into agent decisions.

In [ ]:
class LoggingMiddleware(AgentMiddleware):
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            print("[LOG] Agent input:", inputs)
            result = handler(inputs)
            print("[LOG] Agent output:", result)
            return result
        return wrapped

def dummy_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"output": f"Echo: {inputs.get('input')}"}

logging_mw = LoggingMiddleware(lambda handler: handler)
wrapped_agent = logging_mw(dummy_agent)
print(wrapped_agent({"input": "Hello AI"}))

## 2. Transforming prompts, tool selection, output formatting

Middleware can rewrite prompts, choose which tools are available, and normalize the final output. These transformations allow the agent to receive safer inputs and return cleaner results.

In [ ]:
class PromptTransformMiddleware(AgentMiddleware):
    def __init__(self, transform: Callable[[str], str]):
        self.transform = transform

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            prompt = inputs.get("prompt", "")
            inputs["prompt"] = self.transform(prompt)
            return handler(inputs)
        return wrapped

class OutputFormatMiddleware(AgentMiddleware):
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            result = handler(inputs)
            if "output" in result:
                result["formatted_output"] = result["output"].strip()
            return result
        return wrapped

def basic_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"output": inputs.get("prompt", "") + " [processed]"}

prompt_middleware = PromptTransformMiddleware(lambda prompt: prompt + ' Please answer clearly.')
format_middleware = OutputFormatMiddleware(lambda handler: handler)
wrapped = prompt_middleware(format_middleware(basic_agent))
print(wrapped({"prompt": "What is middleware?"}))

## 3. Retries, fallback, early termination

Retries and fallback logic help make agents robust when external services fail or when the response is not satisfactory. Early termination can stop the agent if a guardrail is triggered.

In [ ]:
class RetryMiddleware(AgentMiddleware):
    def __init__(self, retries: int = 2):
        self.retries = retries

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            last_exception = None
            for attempt in range(1, self.retries + 1):
                try:
                    return handler(inputs)
                except Exception as exc:
                    print(f"[RETRY] attempt {attempt} failed: {exc}")
                    last_exception = exc
            raise last_exception
        return wrapped

class FallbackMiddleware(AgentMiddleware):
    def __init__(self, fallback_response: str):
        self.fallback_response = fallback_response

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            try:
                return handler(inputs)
            except Exception:
                return {"output": self.fallback_response}
        return wrapped

def flaky_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    raise RuntimeError("Model call failed")

handler = RetryMiddleware(retries=3)(FallbackMiddleware("Sorry, I cannot answer right now.")(flaky_agent))
print(handler({"input": "Test"}))

## 4. Rate limits, guardrails, and PII detection

Middleware can enforce request quotas, prevent disallowed content, and detect sensitive data before the agent runs. This is critical for safety and compliance.

In [ ]:
import time
from collections import deque

class RateLimitMiddleware(AgentMiddleware):
    def __init__(self, max_calls: int, period_seconds: int):
        self.max_calls = max_calls
        self.period_seconds = period_seconds
        self.calls = deque()

    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            now = time.time()
            while self.calls and now - self.calls[0] > self.period_seconds:
                self.calls.popleft()
            if len(self.calls) >= self.max_calls:
                return {"output": "Rate limit exceeded. Try again later."}
            self.calls.append(now)
            return handler(inputs)
        return wrapped

class PiiMiddleware(AgentMiddleware):
    def __call__(self, handler: AgentFunction) -> AgentFunction:
        def wrapped(inputs: Dict[str, Any]) -> Dict[str, Any]:
            prompt = inputs.get("prompt", "
,
SSN" in prompt or "credit card" in prompt:
                return {"output": "Potential sensitive data detected. Request blocked."}
            return handler(inputs)
        return wrapped

def simple_agent(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"output": "OK"}

guarded_agent = RateLimitMiddleware(2, 10)(PiiMiddleware(lambda handler: handler)(simple_agent))
print(guarded_agent({"prompt": "Hello"}))
print(guarded_agent({"prompt": "Hello again"}))
print(guarded_agent({"prompt": "Send SSN details"}))

## Summary

Middleware gives you a clean way to add cross-cutting controls to LangChain agents. It helps you observe behavior, transform inputs and outputs, handle failures, and enforce safety without changing the core agent implementation.